In [1]:
# Cell 1 — Imports & description
import os
import re
from glob import glob
from typing import List, Tuple

import numpy as np
import torch
import rasterio
from rasterio.windows import Window
import segmentation_models_pytorch as smp

"""
Batches aligned 2030 SSP rasters by internal blocks, normalizes per channel,
runs the loaded model (optionally with sigmoid), and writes compressed
single-band CISI GeoTIFFs per SSP.
"""


'\nBatches aligned 2030 SSP rasters by internal blocks, normalizes per channel,\nruns the loaded model (optionally with sigmoid), and writes compressed\nsingle-band CISI GeoTIFFs per SSP.\n'

In [2]:
# Cell 2 — Global config

# Year
YEARS = [2030, 2050, 2100]

# Root folder containing SSP1..SSP5 subfolders
DATA_ROOT = "READY_data"

# Where to write CISI projections
OUT_DIR   = "predictions"

# Trained model checkpoint (from training notebook)
CKPT_PATH = "checkpoints/best_unet_regression.pt"

# We only infer 2030 now
YEAR = "2030"
SSP_FOLDERS = [f"SSP{i}" for i in range(1, 6)]

# Apply sigmoid here if the model head has no activation
APPLY_SIGMOID = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32

# Normalization stats (saved from training; see norm_stats.npz export)
NORM_STATS_PATH = "checkpoints/norm_stats.npz"  # .npz with 'mean' and 'std'
FALLBACK_MEAN = None  # optional manual override
FALLBACK_STD  = None

# Optional: cap extreme values like in training
CLAMP_MIN, CLAMP_MAX = -1e6, 1e6

# Batch tiles for speed (works if model is fully convolutional)
BATCH_SIZE = 4


In [3]:
# Cell 3 — Model loader (U-Net regression)

def load_model(in_channels: int) -> torch.nn.Module:
    """
    Rebuild the U-Net used during training and load weights from the checkpoint.
    """
    ckpt = torch.load(CKPT_PATH, map_location="cpu")

    # Try to recover backbone and input channels from checkpoint metadata
    backbone = ckpt.get("backbone", "resnet34")
    ckpt_in_ch = ckpt.get("in_channels", in_channels)

    model = smp.Unet(
        encoder_name=backbone,
        encoder_weights=None,      # no ImageNet pretraining; matches training
        in_channels=ckpt_in_ch,    # same channels as during training
        classes=1,
        activation=None,           # sigmoid applied externally if needed
    )

    # Your training code saved "model_state"
    if "model_state" in ckpt:
        state_dict = ckpt["model_state"]
    elif "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE).eval()
    return model


In [4]:
# Cell 4 — Helper functions: file listing, normalization stats, list year rastering

def natural_key(s: str):
    """Sort strings with embedded numbers: class_2 < class_10."""
    return [int(t) if t.isdigit() else t.lower()
            for t in re.findall(r'\d+|\D+', os.path.basename(s))]

def list_year_rasters(ssp_dir: str, year: int):
    """
    Return predictor rasters in the SAME ORDER as training historic_3band_025deg.tif:
    [LC (0..44), POP (0..101), GDP (large)]
    """
    base = os.path.basename(ssp_dir).upper()
    ssp_num = int("".join(ch for ch in base if ch.isdigit()))

    lc  = os.path.join(ssp_dir, f"SSP{ssp_num}_{year}_LC_025.tif")
    pop = os.path.join(ssp_dir, f"SSP{ssp_num}_{year}_EU_UK_POP_025.tif")
    gdp = os.path.join(ssp_dir, f"GDP{year}__025_ssp{ssp_num}_clipped.tif")

    paths = [lc, pop, gdp]

    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing required raster: {p}")

    return paths


def load_norm_stats(n_channels: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Prefer norm stats saved inside the model checkpoint (guaranteed to match training).
    Fall back to norm_stats.npz only if checkpoint doesn't have them.
    """
    ckpt = torch.load(CKPT_PATH, map_location="cpu")

    if "norm_mean" in ckpt and "norm_std" in ckpt and ckpt["norm_mean"] is not None:
        mean = np.asarray(ckpt["norm_mean"], dtype=np.float32)
        std  = np.asarray(ckpt["norm_std"],  dtype=np.float32)
    elif os.path.isfile(NORM_STATS_PATH):
        stats = np.load(NORM_STATS_PATH)
        mean = stats["mean"].astype(np.float32)
        std  = stats["std"].astype(np.float32)
    else:
        mean = np.zeros(n_channels, dtype=np.float32)
        std  = np.ones(n_channels, dtype=np.float32)

    if mean.shape[0] != n_channels or std.shape[0] != n_channels:
        raise ValueError(
            f"Norm stats (C={mean.shape[0]}) do not match input channels (C={n_channels})."
        )

    std = np.where(std == 0, 1.0, std)
    return mean, std



In [5]:
# Cell 5 — Alignment checks and IO helpers (verbose)

def check_alignment(srcs: List[rasterio.DatasetReader]):
    """
    Ensure all input rasters share CRS, transform, width, and height.
    If not, raise with details.
    """
    base_ds = srcs[0]
    base_name = os.path.basename(base_ds.name)
    crs = base_ds.crs
    transform = base_ds.transform
    width, height = base_ds.width, base_ds.height

    for i, ds in enumerate(srcs[1:], start=1):
        name = os.path.basename(ds.name)
        problems = []
        if ds.crs != crs:
            problems.append("CRS")
        if ds.transform != transform:
            problems.append("transform")
        if ds.width != width or ds.height != height:
            problems.append("shape")

        if problems:
            raise ValueError(
                f"Input rasters are not aligned between '{base_name}' and '{name}' "
                f"(mismatch in: {', '.join(problems)})"
            )

def read_stack_window(srcs, window):
    """
    Read a consistent window from each open raster in `srcs`
    and return a stacked array of shape (C, h, w).
    """
    bands = []
    for ds in srcs:
        arr = ds.read(1, window=window)  # (h, w)
        bands.append(arr)
    x = np.stack(bands, axis=0)          # (C, h, w)
    return x

def write_window(dst, window, arr2d):
    """
    Write a 2D float32 array into output dataset for a given window.
    """
    dst.write(arr2d.astype(np.float32), 1, window=window)

def normalize_batch(x: torch.Tensor, mean, std):
    """
    Channel-wise normalization for a batch.
    x: (B, C, H, W)
    mean/std: length C arrays/lists (from norm_stats.npz)
    """
    mean_t = torch.as_tensor(mean, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    std_t  = torch.as_tensor(std,  device=x.device, dtype=x.dtype).view(1, -1, 1, 1).clamp_min(1e-6)
    return (x - mean_t) / std_t



In [6]:
# Cell 6 — Inference for a single SSP folder

def infer_one_ssp(ssp_path: str, out_path: str, year: int):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Discover 2030 predictor rasters for this SSP
    raster_paths = list_year_rasters(ssp_path, year)
    print("Raster paths returned:")
    for p in raster_paths:
        print(" -", os.path.basename(p))
    print("n_channels =", len(raster_paths))
    n_channels = len(raster_paths)

    # DEBUG: confirm channel order by magnitude (one small window)
    tmp_srcs = [rasterio.open(p) for p in raster_paths]
    _, w0 = next(iter(tmp_srcs[0].block_windows(1)))
    x0 = read_stack_window(tmp_srcs, w0)
    x0 = np.nan_to_num(x0, nan=0.0, posinf=0.0, neginf=0.0)
    print("DEBUG channel magnitudes:")
    for i, p in enumerate(raster_paths):
        a = x0[i]
        print(f"  ch{i} {os.path.basename(p)} min/max:", float(a.min()), float(a.max()))
    for ds in tmp_srcs:
        ds.close()


    # Open sources and validate alignment
    srcs = [rasterio.open(p) for p in raster_paths]
    try:
        check_alignment(srcs)
        ref = srcs[0]
        profile = ref.profile.copy()
        profile.update(
            count=1,
            dtype="float32",
            nodata=None,
            compress="deflate",
            predictor=3,
            zlevel=6,
        )

        # Load normalization and model
        mean, std = load_norm_stats(n_channels)
        print("Using norm mean/std:", mean.tolist(), std.tolist())
        model = load_model(in_channels=n_channels)

        # Debugging
        _, window = next(iter(ref.block_windows(1)))
        x_np = read_stack_window(srcs, window)
        x_np = np.nan_to_num(x_np, nan=0.0, posinf=0.0, neginf=0.0)
        x = torch.from_numpy(x_np).unsqueeze(0).to(DEVICE, dtype=DTYPE)
        x = normalize_batch(x, mean, std)
        with torch.no_grad():
            y_logit = model(x)  # BEFORE sigmoid

        yl = y_logit.squeeze().cpu().numpy()

        print("DEBUG logits:")
        print("  min:", float(np.min(yl)))
        print("  max:", float(np.max(yl)))
        print("  mean:", float(np.mean(yl)))

        print("DEBUG sigmoid(logits):")
        sig = 1 / (1 + np.exp(-yl))
        print("  min:", float(np.min(sig)))
        print("  max:", float(np.max(sig)))
        print("  mean:", float(np.mean(sig)))


        with rasterio.open(out_path, "w", **profile) as dst:
            tiles = []
            tile_windows = []
            current_hw = None

            for _, window in ref.block_windows(1):
                x_np = read_stack_window(srcs, window)  # (C,h,w)

                # Replace NaNs/Infs with 0
                x_np = np.nan_to_num(x_np, nan=0.0, posinf=0.0, neginf=0.0)

                # POP is channel 1 in [LC, POP, GDP]
                pop = x_np[1]
                pop = np.where(pop <= -9999, 0.0, pop)  # catches -99999 etc.
                x_np[1] = pop

                tiles.append(torch.from_numpy(x_np).unsqueeze(0))  # append ONCE
                tile_windows.append(window)

                if len(tiles) == BATCH_SIZE:
                    batch = torch.cat(tiles, dim=0).to(DEVICE, dtype=DTYPE)
                    batch = normalize_batch(batch, mean, std)
                    with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):
                        y = model(batch)
                        if APPLY_SIGMOID:
                            y = torch.sigmoid(y)
                    y_np = y.squeeze(1).cpu().numpy().astype(np.float32)
                    for arr, win in zip(y_np, tile_windows):
                        write_window(dst, win, arr)
                    tiles.clear()
                    tile_windows.clear()


        print(f"[OK] Wrote {out_path}")

    finally:
        for ds in srcs:
            ds.close()



In [7]:
# Cell 7 — Main loop over SSP1..SSP5

def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    for ssp in SSP_FOLDERS:
        for year in [2030, 2050, 2100]:
            ssp_dir = os.path.join(DATA_ROOT, ssp)
            if not os.path.isdir(ssp_dir):
                print(f"[WARN] Skipping missing {ssp_dir}")
                continue
            out_file = os.path.join(OUT_DIR, f"CISI_{year}_{ssp}.tif")
            infer_one_ssp(ssp_dir, out_file, year)

if __name__ == "__main__":
    main()


Raster paths returned:
 - SSP1_2030_LC_025.tif
 - SSP1_2030_EU_UK_POP_025.tif
 - GDP2030__025_ssp1_clipped.tif
n_channels = 3
DEBUG channel magnitudes:
  ch0 SSP1_2030_LC_025.tif min/max: 0.0 7.0
  ch1 SSP1_2030_EU_UK_POP_025.tif min/max: 0.0 0.0
  ch2 GDP2030__025_ssp1_clipped.tif min/max: 0.0 30215.63671875
Using norm mean/std: [6.364697456359863, 1.1613380908966064, 19452.46875] [11.615069389343262, 4.14699125289917, 106526.6875]
DEBUG logits:
  min: -8.21243953704834
  max: -1.90442955493927
  mean: -5.110295295715332
DEBUG sigmoid(logits):
  min: 0.0002711846027523279
  max: 0.12960796058177948
  mean: 0.014540267176926136


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_17956\4136602563.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):


[OK] Wrote predictions\CISI_2030_SSP1.tif
Raster paths returned:
 - SSP1_2050_LC_025.tif
 - SSP1_2050_EU_UK_POP_025.tif
 - GDP2050__025_ssp1_clipped.tif
n_channels = 3
DEBUG channel magnitudes:
  ch0 SSP1_2050_LC_025.tif min/max: 0.0 7.0
  ch1 SSP1_2050_EU_UK_POP_025.tif min/max: 0.0 0.0
  ch2 GDP2050__025_ssp1_clipped.tif min/max: 0.0 27614.8203125
Using norm mean/std: [6.364697456359863, 1.1613380908966064, 19452.46875] [11.615069389343262, 4.14699125289917, 106526.6875]
DEBUG logits:
  min: -8.20624828338623
  max: -1.9045495986938477
  mean: -5.109590530395508
DEBUG sigmoid(logits):
  min: 0.00027286834665574133
  max: 0.12959441542625427
  mean: 0.014544128440320492
[OK] Wrote predictions\CISI_2050_SSP1.tif
Raster paths returned:
 - SSP1_2100_LC_025.tif
 - SSP1_2100_EU_UK_POP_025.tif
 - GDP2100__025_ssp1_clipped.tif
n_channels = 3
DEBUG channel magnitudes:
  ch0 SSP1_2100_LC_025.tif min/max: 0.0 7.0
  ch1 SSP1_2100_EU_UK_POP_025.tif min/max: 0.0 0.0
  ch2 GDP2100__025_ssp1_clipped

In [8]:
import rasterio
import numpy as np

with rasterio.open("predictions/CISI_2030_SSP1.tif") as ds:
    x = ds.read(1)
    x = x[np.isfinite(x)]

print("PRED min/max/mean:", float(x.min()), float(x.max()), float(x.mean()))
print("PRED percentiles:", [float(np.percentile(x, p)) for p in [50, 90, 95, 99, 99.9]])
print("fraction > 0:", float((x > 0).mean()))
print("fraction > 0.01:", float((x > 0.01).mean()))


PRED min/max/mean: 0.0 0.1296079158782959 0.0029550467152148485
PRED percentiles: [0.0, 0.007390326354652643, 0.015955191105604172, 0.06295757740736008, 0.09635207056999207]
fraction > 0: 0.21097847712063664
fraction > 0.01: 0.0809594863447278


In [9]:
with rasterio.open("READY_data\labels/2024_CISI_025deg.tif") as ds:
    y = ds.read(1)
    y = y[np.isfinite(y)]

print("LABEL min/max/mean:", float(y.min()), float(y.max()), float(y.mean()))
print("LABEL percentiles:", [float(np.percentile(y, p)) for p in [50, 90, 95, 99, 99.9]])
print("fraction > 0:", float((y > 0).mean()))


LABEL min/max/mean: 0.0 0.7352721095085144 0.03292354196310043
LABEL percentiles: [0.02038705348968506, 0.08082064986228943, 0.11008686572313309, 0.18895000219345093, 0.34734460711479187]
fraction > 0: 0.9325338335789897


In [29]:
import rasterio
import numpy as np

label_path = "READY_data/labels/2024_CISI_025deg.tif"  # adjust if needed

with rasterio.open(label_path) as ds:
    arr = ds.read(1).astype(np.float32)

arr = arr[np.isfinite(arr)]

print("Label raster stats:")
print("  shape:", arr.shape)
print("  min/max:", float(arr.min()), float(arr.max()))
print("  mean:", float(arr.mean()))
print("  percentiles:", {
    "p50": float(np.percentile(arr, 50)),
    "p90": float(np.percentile(arr, 90)),
    "p95": float(np.percentile(arr, 95)),
    "p99": float(np.percentile(arr, 99)),
})
print("  fraction > 0:", float((arr > 0).mean()))


Label raster stats:
  shape: (14926,)
  min/max: 0.0 0.7352721095085144
  mean: 0.03292354196310043
  percentiles: {'p50': 0.02038705348968506, 'p90': 0.08082064986228943, 'p95': 0.11008686572313309, 'p99': 0.18895000219345093}
  fraction > 0: 0.9325338335789897


# Check if the label raster is okay or not

In [10]:
import pandas as pd
import numpy as np

df = pd.read_feather("READY_data/labels/2024_CISI_clipped_025.feather")  # adjust if needed

print("Columns:", df.columns.tolist())
print("CISI min/max/mean:", float(df["CISI"].min()), float(df["CISI"].max()), float(df["CISI"].mean()))
print("CISI percentiles:", [float(np.percentile(df["CISI"].values, q)) for q in [50,90,95,99,99.9]])
print("Nonzero count:", int((df["CISI"]>0).sum()), "/", len(df))


Columns: ['CISI', 'Subscore_energy', 'Subscore_transportation', 'Subscore_water', 'Subscore_waste', 'Subscore_telecommunication', 'Subscore_healthcare', 'Subscore_education', 'geometry']
CISI min/max/mean: 0.0 0.7352721346115842 0.032823066653059
CISI percentiles: [0.020258890268890176, 0.08066019529501328, 0.10993924118546466, 0.18854921220425921, 0.3470607571538749]
Nonzero count: 14005 / 15023


### Checks

In [11]:
import os
import numpy as np
import rasterio

YEARS = [2030, 2050, 2100]
SSP_FOLDERS = [f"SSP{i}" for i in range(1, 6)]

def stats_raster(path):
    with rasterio.open(path) as ds:
        a = ds.read(1).astype("float32")
        nan = int(np.isnan(a).sum())
        return {
            "shape": (ds.height, ds.width),
            "nan": nan,
            "min": float(np.nanmin(a)),
            "max": float(np.nanmax(a)),
            "mean": float(np.nanmean(a)),
        }

for year in YEARS:
    print(f"\n==== CISI predictions {year} ====")
    for ssp in SSP_FOLDERS:
        p = os.path.join(OUT_DIR, f"CISI_{year}_{ssp}.tif")
        if not os.path.exists(p):
            print(f"{ssp}: missing {p}")
            continue
        st = stats_raster(p)
        print(f"{ssp}: min/max {st['min']:.6f} {st['max']:.6f} mean {st['mean']:.6f} nan {st['nan']}")



==== CISI predictions 2030 ====
SSP1: min/max 0.000000 0.129608 mean 0.002955 nan 0
SSP2: min/max 0.000000 0.129613 mean 0.002955 nan 0
SSP3: min/max 0.000000 0.129630 mean 0.002963 nan 0
SSP4: min/max 0.000000 0.129614 mean 0.002959 nan 0
SSP5: min/max 0.000000 0.129593 mean 0.002954 nan 0

==== CISI predictions 2050 ====
SSP1: min/max 0.000000 0.129594 mean 0.002929 nan 0
SSP2: min/max 0.000000 0.129598 mean 0.002929 nan 0
SSP3: min/max 0.000000 0.129623 mean 0.002944 nan 0
SSP4: min/max 0.000000 0.129598 mean 0.002919 nan 0
SSP5: min/max 0.000000 0.129559 mean 0.002855 nan 0

==== CISI predictions 2100 ====
SSP1: min/max 0.000000 0.129603 mean 0.002811 nan 0
SSP2: min/max 0.000000 0.129605 mean 0.002755 nan 0
SSP3: min/max 0.000000 0.129643 mean 0.002799 nan 0
SSP4: min/max 0.000000 0.129622 mean 0.002773 nan 0
SSP5: min/max 0.000000 0.134395 mean 0.002666 nan 0


In [12]:
import os
import numpy as np
import rasterio

def read1(path):
    with rasterio.open(path) as ds:
        a = ds.read(1).astype("float32")
    return a

def masked_stats(a):
    m = np.isfinite(a)
    if m.sum() == 0:
        return {"valid": 0, "mean": np.nan, "std": np.nan, "min": np.nan, "max": np.nan}
    return {
        "valid": int(m.sum()),
        "mean": float(np.nanmean(a)),
        "std": float(np.nanstd(a)),
        "min": float(np.nanmin(a)),
        "max": float(np.nanmax(a)),
    }

def mean_abs_diff(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() == 0:
        return np.nan, 0
    return float(np.mean(np.abs(a[m] - b[m]))), int(m.sum())

for year in [2030, 2050, 2100]:
    print(f"\n==== Inputs check SSP1 vs SSP5 for {year} ====")

    ssp1 = rf"READY_data/SSP1"
    ssp5 = rf"READY_data/SSP5"

    lc1 = os.path.join(ssp1, f"SSP1_{year}_LC_025.tif")
    lc5 = os.path.join(ssp5, f"SSP5_{year}_LC_025.tif")
    pop1 = os.path.join(ssp1, f"SSP1_{year}_EU_UK_POP_025.tif")
    pop5 = os.path.join(ssp5, f"SSP5_{year}_EU_UK_POP_025.tif")
    gdp1 = os.path.join(ssp1, f"GDP{year}__025_ssp1_clipped.tif")
    gdp5 = os.path.join(ssp5, f"GDP{year}__025_ssp5_clipped.tif")

    for label, p1, p5 in [("LC", lc1, lc5), ("POP", pop1, pop5), ("GDP", gdp1, gdp5)]:
        a = read1(p1); b = read1(p5)
        sa = masked_stats(a); sb = masked_stats(b)
        mad, n = mean_abs_diff(a, b)
        print(f"{label} {year} SSP1 vs SSP5")
        print(f"  A mean/std {sa['mean']:.3f} {sa['std']:.3f} min/max {sa['min']:.3f} {sa['max']:.3f} valid {sa['valid']}")
        print(f"  B mean/std {sb['mean']:.3f} {sb['std']:.3f} min/max {sb['min']:.3f} {sb['max']:.3f} valid {sb['valid']}")
        print(f"  mean|A-B|: {mad:.6f} (n={n})")



==== Inputs check SSP1 vs SSP5 for 2030 ====
LC 2030 SSP1 vs SSP5
  A mean/std 1.201 1.882 min/max 0.000 7.000 valid 44232
  B mean/std 1.211 1.893 min/max 0.000 7.000 valid 44232
  mean|A-B|: 0.055412 (n=44232)
POP 2030 SSP1 vs SSP5
  A mean/std 63.094 180.960 min/max 0.000 5817.995 valid 11070
  B mean/std 63.930 184.205 min/max 0.000 5880.740 valid 11070
  mean|A-B|: 1.060479 (n=11070)
GDP 2030 SSP1 vs SSP5
  A mean/std 531000704.000 4389662720.000 min/max 0.000 344021827584.000 valid 44232
  B mean/std 548094848.000 4527526400.000 min/max 0.000 353885257728.000 valid 44232
  mean|A-B|: 17301808.000000 (n=44232)

==== Inputs check SSP1 vs SSP5 for 2050 ====
LC 2050 SSP1 vs SSP5
  A mean/std 1.206 1.893 min/max 0.000 7.000 valid 44232
  B mean/std 1.212 1.895 min/max 0.000 7.000 valid 44232
  mean|A-B|: 0.061743 (n=44232)
POP 2050 SSP1 vs SSP5
  A mean/std 62.983 184.386 min/max 0.000 6154.038 valid 11070
  B mean/std 66.424 198.071 min/max 0.000 6430.964 valid 11070
  mean|A-B|: 3.

In [13]:
import numpy as np
import rasterio

p = "predictions/CISI_2030_SSP1.tif"
with rasterio.open(p) as ds:
    a = ds.read(1)

print("min/max", float(a.min()), float(a.max()))
print("mean", float(a.mean()))
print("zeros", int((a==0).sum()), "/", a.size)

for q in [90, 95, 99, 99.5, 99.9]:
    print(f"p{q} =", float(np.percentile(a, q)))

for thr in [1e-6, 1e-4, 1e-3, 1e-2, 5e-2]:
    print(f"frac > {thr}: {(a>thr).mean():.6f}")


min/max 0.0 0.1296079158782959
mean 0.0029550467152148485
zeros 34900 / 44232
p90 = 0.007390326354652643
p95 = 0.015955191105604172
p99 = 0.06295757740736008
p99.5 = 0.08324022591114044
p99.9 = 0.09635207056999207
frac > 1e-06: 0.201460
frac > 0.0001: 0.198748
frac > 0.001: 0.174399
frac > 0.01: 0.080959
frac > 0.05: 0.017928


In [14]:
import numpy as np, rasterio

pred_path  = "predictions/CISI_hist.tif"          # or CISI_2030_SSP1.tif
label_path = "READY_data/labels/2024_CISI_025deg.tif"  # adjust

with rasterio.open(pred_path) as pds, rasterio.open(label_path) as lds:
    p = pds.read(1)
    y = lds.read(1)

m = np.isfinite(y)
p = p[m]; y = y[m]

print("PRED min/max/mean:", float(p.min()), float(p.max()), float(p.mean()))
print("LABEL min/max/mean:", float(y.min()), float(y.max()), float(y.mean()))
print("LABEL percentiles:", [float(np.percentile(y, q)) for q in [50,90,95,99,99.9]])
print("PRED percentiles :", [float(np.percentile(p, q)) for q in [50,90,95,99,99.9]])


PRED min/max/mean: 0.0 3.5553369352632086e-26 2.3819920360090946e-30
LABEL min/max/mean: 0.0 0.7352721095085144 0.03292354196310043
LABEL percentiles: [0.02038705348968506, 0.08082064986228943, 0.11008686572313309, 0.18895000219345093, 0.34734460711479187]
PRED percentiles : [0.0, 0.0, 0.0, 0.0, 0.0]


# Checks

In [15]:
def list_historical_rasters(input_dir: str):
    """
    MUST match training order:
    [GDP, POP, LANDCOVER] at 0.25°
    """
    gdp = os.path.join(input_dir, "2019_gdp_aligned_025deg.tif")
    pop = os.path.join(input_dir, "2020_pop_aligned_025deg.tif")
    lc  = os.path.join(input_dir, "2018_landcover_aligned_025deg.tif")

    paths = [gdp, pop, lc]

    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing historical raster: {p}")

    return paths


In [16]:
def infer_historical(out_path: str):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    raster_paths = list_historical_rasters("READY_data/inputs")
    print("Historical rasters:")
    for p in raster_paths:
        print(" -", os.path.basename(p))

    n_channels = len(raster_paths)

    srcs = [rasterio.open(p) for p in raster_paths]
    try:
        check_alignment(srcs)
        ref = srcs[0]

        profile = ref.profile.copy()
        profile.update(
            count=1,
            dtype="float32",
            nodata=None,
            compress="deflate",
        )

        mean, std = load_norm_stats(n_channels)
        model = load_model(in_channels=n_channels)
        model = model.float()

        with rasterio.open(out_path, "w", **profile) as dst:
            tiles, windows = [], []

            for _, window in ref.block_windows(1):
                x = read_stack_window(srcs, window)
                x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

                x = torch.from_numpy(x).unsqueeze(0).to(DEVICE, dtype=torch.float32)
                x = normalize_batch(x, mean, std)

                with torch.no_grad():
                    y = model(x)
                    if APPLY_SIGMOID:
                        y = torch.sigmoid(y)

                y_np = y.squeeze().cpu().numpy().astype(np.float32)
                write_window(dst, window, y_np)

            if tiles:
                batch = torch.cat(tiles).to(DEVICE)
                batch = normalize_batch(batch, mean, std)
                with torch.no_grad():
                    y = model(batch)
                    if APPLY_SIGMOID:
                        y = torch.sigmoid(y)
                y_np = y.squeeze(1).cpu().numpy()
                for arr, win in zip(y_np, windows):
                    write_window(dst, win, arr)

        print(f"[OK] Wrote {out_path}")

    finally:
        for ds in srcs:
            ds.close()


In [17]:
infer_historical("predictions/CISI_hist.tif")


Historical rasters:
 - 2019_gdp_aligned_025deg.tif
 - 2020_pop_aligned_025deg.tif
 - 2018_landcover_aligned_025deg.tif


[OK] Wrote predictions/CISI_hist.tif


In [18]:
with rasterio.open("predictions/CISI_hist.tif") as ds:
    arr = ds.read(1)
    print("shape:", arr.shape)
    print("min:", float(np.nanmin(arr)))
    print("max:", float(np.nanmax(arr)))
    print("mean:", float(np.nanmean(arr)))
    print("nan count:", np.isnan(arr).sum())


shape: (195, 228)
min: 0.0
max: 0.07255081832408905
mean: 8.64448429638287e-06
nan count: 0


In [19]:
# # Debug cell — inspect rasters for one SSP

# import rasterio

# def inspect_ssp(ssp_name: str = "SSP1"):
#     ssp_dir = os.path.join(DATA_ROOT, ssp_name)
#     raster_paths = list_2030_rasters(ssp_dir)
#     print(f"2030 rasters for {ssp_name}:")
#     for i, p in enumerate(raster_paths):
#         with rasterio.open(p) as ds:
#             print(f"\n[{i}] {os.path.basename(p)}")
#             print("  CRS:      ", ds.crs)
#             print("  Transform:", ds.transform)
#             print("  Size:     ", ds.width, "x", ds.height)

# inspect_ssp("SSP1")
# print(DATA_ROOT)
# print(ssp_name)

In [20]:
# # Debug cell — inspect rasters for one SSP for a given YEAR (2050/2100)
# # (Same style as your 2030 inspector: just prints CRS/Transform/Size for each raster)

# import os
# import rasterio

# # Set this to the folder that contains the SSP1..SSP5 folders
# DATA_ROOT = r"READY_data"

# def list_year_rasters(ssp_dir: str, year: int):
#     """
#     Minimal variant of list_2030_rasters, but for any year.
#     It returns all .tif files in the SSP folder that contain the YEAR in the filename.
#     """
#     return sorted(
#         os.path.join(ssp_dir, f)
#         for f in os.listdir(ssp_dir)
#         if f.lower().endswith("025.tif") and str(year) in f
#     )

# def inspect_ssp_year(ssp_name: str = "SSP1", year: int = 2050):
#     # works for folders named SSP1 or ssp1
#     ssp_dir = os.path.join(DATA_ROOT, ssp_name)
#     if not os.path.isdir(ssp_dir):
#         ssp_dir = os.path.join(DATA_ROOT, ssp_name.lower())

#     raster_paths = list_year_rasters(ssp_dir, year)
#     print(f"{year} rasters for {ssp_name}:")
#     for i, p in enumerate(raster_paths):
#         with rasterio.open(p) as ds:
#             print(f"\n[{i}] {os.path.basename(p)}")
#             print("  CRS:      ", ds.crs)
#             print("  Transform:", ds.transform)
#             print("  Size:     ", ds.width, "x", ds.height)

# # Run for 2050 / 2100
# inspect_ssp_year("SSP1", 2050)
# inspect_ssp_year("SSP1", 2100)


In [21]:
# # Regrid POP (_01) to match GDP 0.25° grid, and write a new POP file ending with _025.tif
# # Works for one SSP folder and one year.

# import os
# import numpy as np
# import rasterio
# from rasterio.warp import reproject, Resampling

# def make_pop_025(ssp_dir: str, ssp_num: int, year: int):
#     gdp_path = os.path.join(ssp_dir, f"GDP{year}__025_ssp{ssp_num}_clipped.tif")
#     pop_01   = os.path.join(ssp_dir, f"SSP{ssp_num}_{year}_EU_UK_POP_01.tif")
#     pop_025  = os.path.join(ssp_dir, f"SSP{ssp_num}_{year}_EU_UK_POP_025.tif")

#     if not os.path.exists(gdp_path):
#         raise FileNotFoundError(f"Missing GDP reference: {gdp_path}")
#     if not os.path.exists(pop_01):
#         raise FileNotFoundError(f"Missing POP source (_01): {pop_01}")

#     with rasterio.open(gdp_path) as ref:
#         ref_profile   = ref.profile.copy()
#         ref_crs       = ref.crs
#         ref_transform = ref.transform
#         H, W          = ref.height, ref.width

#     # Output profile: single band float32, keep the reference grid exactly
#     ref_profile.update(count=1, dtype="float32", nodata=np.nan)

#     dst = np.full((H, W), np.nan, dtype=np.float32)

#     with rasterio.open(pop_01) as src:
#         src_data = src.read(1).astype(np.float32)
#         reproject(
#             source=src_data,
#             destination=dst,
#             src_transform=src.transform,
#             src_crs=src.crs,
#             dst_transform=ref_transform,
#             dst_crs=ref_crs,
#             resampling=Resampling.bilinear,  # continuous variable (population)
#         )

#     with rasterio.open(pop_025, "w", **ref_profile) as out:
#         out.write(dst, 1)

#     # Verify
#     with rasterio.open(pop_025) as a, rasterio.open(gdp_path) as b:
#         ok = (a.crs == b.crs) and (a.transform == b.transform) and (a.width == b.width) and (a.height == b.height)

#     print("Wrote:", pop_025)
#     print("Aligned to GDP grid:", ok)
#     if not ok:
#         raise RuntimeError("POP_025 was written but is not perfectly aligned to the GDP reference.")

#     return pop_025



In [22]:
# pop_025_path = make_pop_025(r"READY_data/SSP5", ssp_num=5, year=2030)
# pop_025_path = make_pop_025(r"READY_data/SSP5", ssp_num=5, year=2050)
# pop_025_path = make_pop_025(r"READY_data/SSP5", ssp_num=5, year=2100)


In [23]:
import os
ssp_dir = r"READY_data/SSP1"
for f in sorted(os.listdir(ssp_dir)):
    if "2030" in f and f.lower().endswith(".tif"):
        print(f)

GDP2030__025_ssp1_clipped.tif
SSP1_2030_EU_UK_POP_01.tif
SSP1_2030_EU_UK_POP_025.tif
SSP1_2030_LC_025.tif
SSP1_RCP26_2030_class_1_clipped.tif
SSP1_RCP26_2030_class_2_clipped.tif
SSP1_RCP26_2030_class_3_clipped.tif
SSP1_RCP26_2030_class_4_clipped.tif
SSP1_RCP26_2030_class_5_clipped.tif
SSP1_RCP26_2030_class_6_clipped.tif
SSP1_RCP26_2030_class_7_clipped.tif


In [24]:
# Check if pop file was correctly transformed

In [25]:
import rasterio, numpy as np

p = r"READY_data/SSP1/SSP1_2030_EU_UK_POP_025.tif"
with rasterio.open(p) as ds:
    a = ds.read(1).astype("float32")
    print("nodata:", ds.nodata)
    print("dtype:", a.dtype)
    print("min/max:", float(np.nanmin(a)), float(np.nanmax(a)))
    print("count nan:", int(np.isnan(a).sum()), "/", a.size)
    if ds.nodata is not None:
        print("count == nodata:", int((a == ds.nodata).sum()), "/", a.size)
    print("count < 0:", int((a < 0).sum()), "/", a.size)


nodata: nan
dtype: float32
min/max: 0.0 5817.99462890625
count nan: 33162 / 44232
count == nodata: 0 / 44232
count < 0: 0 / 44232


In [26]:
import rasterio, numpy as np

p = r"READY_data\SSP1\SSP1_2030_EU_UK_POP_01.tif"  # <-- change to your 0.1° POP path
with rasterio.open(p) as ds:
    a = ds.read(1).astype("float32")
    print("nodata:", ds.nodata)
    print("min/max:", float(np.nanmin(a)), float(np.nanmax(a)))
    print("nan count:", int(np.isnan(a).sum()), "/", a.size)
    if ds.nodata is not None and not np.isnan(ds.nodata):
        print("count==nodata:", int((a == ds.nodata).sum()))
    print("count<0:", int((a < 0).sum()))


nodata: -99999.0
min/max: -99999.0 30528.986328125
nan count: 0 / 38617920
count==nodata: 30108118
count<0: 30108118


In [27]:
import rasterio, numpy as np

p = r"READY_data/SSP1/SSP1_2030_EU_UK_POP_01.tif"
with rasterio.open(p) as ds:
    a = ds.read(1).astype("float32")
    nod = ds.nodata
    a = np.where(a == nod, np.nan, a)

valid_frac = np.isfinite(a).mean()
print("valid fraction:", valid_frac)
print("valid min/max:", float(np.nanmin(a)), float(np.nanmax(a)))


valid fraction: 0.22035889038042442
valid min/max: 0.0 30528.986328125


In [28]:
import rasterio, numpy as np, os

paths = [
    r"READY_data/SSP1/GDP2030__025_ssp1_clipped.tif",
    r"READY_data/SSP1/SSP1_2030_EU_UK_POP_025.tif",
    r"READY_data/SSP1/SSP1_2030_LC_025.tif",
]

for p in paths:
    with rasterio.open(p) as ds:
        a = ds.read(1).astype("float32")
        a = np.where(np.isfinite(a), a, np.nan)
        print(os.path.basename(p))
        print("  min/max (nan-safe):", float(np.nanmin(a)), float(np.nanmax(a)))
        print("  nan count:", int(np.isnan(a).sum()), "/", a.size)
        # POP check (if you expect 0..6000 and lots of nan)
        print("  neg count:", int((a < 0).sum()))


GDP2030__025_ssp1_clipped.tif
  min/max (nan-safe): 0.0 344021827584.0
  nan count: 0 / 44232
  neg count: 0
SSP1_2030_EU_UK_POP_025.tif
  min/max (nan-safe): 0.0 5817.99462890625
  nan count: 33162 / 44232
  neg count: 0
SSP1_2030_LC_025.tif
  min/max (nan-safe): 0.0 7.0
  nan count: 0 / 44232
  neg count: 0
